In [ ]:
# Package import

import requests
import pandas as pd

# (Azért, hogy pd.Dataframe-nél ne csak az általa választott oszlopmennyiséget adja meg, ezért szükség van az alábbira:)
pd.set_option("display.max_columns",30)
from datetime import datetime

# Checking difference between dates

from dateutil.relativedelta import relativedelta

from typing import Dict

In [ ]:
"""
    Steps for the Transform Load Lambda function
    1. Get data from S3 (taxi, weather)
    2. Weather data transformations --> Done
    3. Taxi data transformations --> Done
    4. Update dim_payment_type --> Done
    5. Update dim_company --> Done
    6. Update fact_taxi_trips with the ids from the dim_payment_type and dim_company --> DONE
    7. Upload dim_weather to S3
    8. Upload fact_taxi_trips to S3
    9. Upload dim_payment_type and dim_company (current and previous versions)

"""

'\n    Steps for the Transform Load Lambda function\n    1. Get data from S3 (taxi, weather)\n    2. Weather data transformations\n    3. Taxi data transformations --> Done\n    4. Update dim_payment_type --> Done\n    5. Update dim_company --> Done\n    6. Update fact_taxi_trips with the ids from the dim_payment_type and dim_company\n    7. Upload dim_weather to S3\n    8. Upload fact_taxi_trips to S3\n    9. Upload dim_payment_type and dim_company (current and previous versions)\n\n'

# Taxi data load

In [ ]:
current_datetime = datetime.now() - relativedelta(months=2)

formatted_datetime = current_datetime.strftime("%Y-%m-%d")

# Taxi trips for 1 specified day with constant date change (code part from 03 notebook)

url = (
    f"https://data.cityofchicago.org/resource/ajtu-isnz.json?"
    f"$where=trip_start_timestamp >= '{formatted_datetime}T00:00:00'"
    f"AND trip_start_timestamp <= '{formatted_datetime}T23:59:59'"
    f"&$limit=30000"
)

response = requests.get(url)
data = response.json()

# Making pd dataframe

taxi_trips = pd.DataFrame(data)

taxi_trips.head()

,trip_id,taxi_id,trip_start_timestamp,trip_end_timestamp,trip_seconds,trip_miles,pickup_community_area,dropoff_community_area,fare,tips,tolls,extras,trip_total,payment_type,company,pickup_centroid_latitude,pickup_centroid_longitude,pickup_centroid_location,dropoff_centroid_latitude,dropoff_centroid_longitude,dropoff_centroid_location,pickup_census_tract,dropoff_census_tract
0,64451d9e83f94dc7cd493a58b21907f723a4d11c,48d462bcf24c2aadcda3fcf689d3f63fc178f23dc73bba...,2025-09-30T23:45:00.000,2025-10-01T00:15:00.000,1563,17.51,76,8,43.25,9.55,0,4,57.3,Credit Card,Sun Taxi,41.980264315,-87.913624596,"{'type': 'Point', 'coordinates': [-87.91362459...",41.899602111,-87.633308037,"{'type': 'Point', 'coordinates': [-87.63330803...",NaN,NaN
1,013cfcf119425927ffaa918d90c6fa14e99fb66f,73b2f5adecea91eeef3900303a07f1b0519a594cffb6b0...,2025-09-30T23:45:00.000,2025-10-01T00:00:00.000,480,1.32,8,32,7.67,0,0,0,8.17,Mobile,Chicago Taxicab,41.899602111,-87.633308037,"{'type': 'Point', 'coordinates': [-87.63330803...",41.878865584,-87.625192142,"{'type': 'Point', 'coordinates': [-87.62519214...",NaN,NaN
2,0c93fd68ba8a33329f43b5ecd06c9adb4bd6c1ca,63d895bf335c522af83a8f2c608e31bb46a5d78cde4df2...,2025-09-30T23:45:00.000,2025-10-01T00:15:00.000,1475,18.34,76,8,45.25,11.15,0,10,66.9,Mobile,Blue Ribbon Taxi Association,41.980264315,-87.913624596,"{'type': 'Point', 'coordinates': [-87.91362459...",41.899602111,-87.633308037,"{'type': 'Point', 'coordinates': [-87.63330803...",NaN,NaN
3,0fb91fa81f44cfd13beef5a9eab75da222dd3214,99ec13d5d806f5f5fa7a57910f8e38d84f90630529f2f8...,2025-09-30T23:45:00.000,2025-10-01T00:00:00.000,249,0.55,32,8,5,0,0,0,5,Cash,5 Star Taxi,41.878865584,-87.625192142,"{'type': 'Point', 'coordinates': [-87.62519214...",41.899602111,-87.633308037,"{'type': 'Point', 'coordinates': [-87.63330803...",NaN,NaN
4,1528b5af92997e65634466efa11c894379843989,58a93192d6a7e977c8c9b10d36ed0e52325afc4b5ece70...,2025-09-30T23:45:00.000,2025-10-01T00:00:00.000,906,5.65,8,5,15.65,4.02,0,0,20.17,Mobile,Taxicab Insurance Agency Llc,41.899602111,-87.633308037,"{'type': 'Point', 'coordinates': [-87.63330803...",41.947791586,-87.683834942,"{'type': 'Point', 'coordinates': [-87.68383494...",NaN,NaN


# Taxi data transformations 

In [ ]:
# Drop non-required columns

taxi_trips.drop(['pickup_census_tract' , 'dropoff_census_tract','pickup_centroid_location','dropoff_centroid_location'], axis=1, inplace=True)

# Drop empty rows

taxi_trips.dropna(inplace=True)

# Rename columns

taxi_trips.rename(columns={'pickup_community_area':'pickup_community_area_id',
                            'dropoff_community_area' : 'dropoff_community_area_id'}, inplace=True )

# Taxi trips startdate datatype from object to datetime

taxi_trips['trip_start_timestamp'] = pd.to_datetime(taxi_trips['trip_start_timestamp'])

# Additional column for weather data --> datetime column rounded for the nearest hour. (based on 06 notebook)

taxi_trips['datetime_for_weather'] = taxi_trips["trip_start_timestamp"].dt.floor('h')

## Organizing above code into one function

In [ ]:
def taxi_trips_transformations(taxi_trips:pd.DataFrame) -> pd.DataFrame:

    """
    Perform transformations on the taxi data.
    1. Drop selected columns
    2. Drop NULL values across all columns
    3. Rename selected columns
    4. Create "datetime_for_weather" helper column (for dim_weather join).

    :param taxi_trips: The DataFrame holding the daily taxi trips.
    :raises TypeError: When taxi_trips parameter is not a valid pandas DataFrame.
    :return:           Transformed taxi trips DataFrame

    """

    # Error handling (REQUIRED ALWAYS)

    if not isinstance(taxi_trips,pd.DataFrame):
        raise TypeError("taxi_trips is not a valid pandas DataFrame.")

        taxi_trips.drop(['pickup_census_tract' , 'dropoff_census_tract','pickup_centroid_location','dropoff_centroid_location'], axis=1, inplace=True)

        # Drop empty rows

        taxi_trips.dropna(inplace=True)

        # Rename columns

        taxi_trips.rename(columns={'pickup_community_area':'pickup_community_area_id',
                                'dropoff_community_area' : 'dropoff_community_area_id'}, inplace=True )

        # Taxi trips startdate datatype from object to datetime

        taxi_trips['trip_start_timestamp'] = pd.to_datetime(taxi_trips['trip_start_timestamp'])

        # Additional column for weather data --> datetime column rounded for the nearest hour. (based on 06 notebook)

        taxi_trips['datetime_for_weather'] = taxi_trips["trip_start_timestamp"].dt.floor('h')

    return taxi_trips

In [ ]:
# Check

taxi_trips_transformed = taxi_trips_transformations(taxi_trips)

taxi_trips_transformed.head()

,trip_id,taxi_id,trip_start_timestamp,trip_end_timestamp,trip_seconds,trip_miles,pickup_community_area_id,dropoff_community_area_id,fare,tips,tolls,extras,trip_total,payment_type,company,pickup_centroid_latitude,pickup_centroid_longitude,dropoff_centroid_latitude,dropoff_centroid_longitude,datetime_for_weather
0,64451d9e83f94dc7cd493a58b21907f723a4d11c,48d462bcf24c2aadcda3fcf689d3f63fc178f23dc73bba...,2025-09-30 23:45:00,2025-10-01T00:15:00.000,1563,17.51,76,8,43.25,9.55,0,4,57.3,Credit Card,Sun Taxi,41.980264315,-87.913624596,41.899602111,-87.633308037,2025-09-30 23:00:00
1,013cfcf119425927ffaa918d90c6fa14e99fb66f,73b2f5adecea91eeef3900303a07f1b0519a594cffb6b0...,2025-09-30 23:45:00,2025-10-01T00:00:00.000,480,1.32,8,32,7.67,0,0,0,8.17,Mobile,Chicago Taxicab,41.899602111,-87.633308037,41.878865584,-87.625192142,2025-09-30 23:00:00
2,0c93fd68ba8a33329f43b5ecd06c9adb4bd6c1ca,63d895bf335c522af83a8f2c608e31bb46a5d78cde4df2...,2025-09-30 23:45:00,2025-10-01T00:15:00.000,1475,18.34,76,8,45.25,11.15,0,10,66.9,Mobile,Blue Ribbon Taxi Association,41.980264315,-87.913624596,41.899602111,-87.633308037,2025-09-30 23:00:00
3,0fb91fa81f44cfd13beef5a9eab75da222dd3214,99ec13d5d806f5f5fa7a57910f8e38d84f90630529f2f8...,2025-09-30 23:45:00,2025-10-01T00:00:00.000,249,0.55,32,8,5,0,0,0,5,Cash,5 Star Taxi,41.878865584,-87.625192142,41.899602111,-87.633308037,2025-09-30 23:00:00
4,1528b5af92997e65634466efa11c894379843989,58a93192d6a7e977c8c9b10d36ed0e52325afc4b5ece70...,2025-09-30 23:45:00,2025-10-01T00:00:00.000,906,5.65,8,5,15.65,4.02,0,0,20.17,Mobile,Taxicab Insurance Agency Llc,41.899602111,-87.633308037,41.947791586,-87.683834942,2025-09-30 23:00:00


In [ ]:
# Check

taxi_trips_transformed.info()

<class 'pandas.core.frame.DataFrame'>
Index: 22580 entries, 0 to 24814
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   trip_id                     22580 non-null  object        
 1   taxi_id                     22580 non-null  object        
 2   trip_start_timestamp        22580 non-null  datetime64[ns]
 3   trip_end_timestamp          22580 non-null  object        
 4   trip_seconds                22580 non-null  object        
 5   trip_miles                  22580 non-null  object        
 6   pickup_community_area_id    22580 non-null  object        
 7   dropoff_community_area_id   22580 non-null  object        
 8   fare                        22580 non-null  object        
 9   tips                        22580 non-null  object        
 10  tolls                       22580 non-null  object        
 11  extras                      22580 non-null  object        


## Dim Company and Dim Payment Type update

In [ ]:
def update_dim_company_dim_payment_type(taxi_trips: pd.DataFrame, dim_df: pd.DataFrame, id_col: str, value_col=str) -> pd.DataFrame:

    """
    Extend the dimension DataFrame with new values if there are any.

    :param taxi_trips: DataFrame with the daily taxi trips.
    :param dim_df: DataFrame with the dimension data (company, payment_type).
    :param id_col: The id column of the dimension DataFrame.
    :param value_col: Name of the column in dimension DataFrame containing the values.
    :return: The updated dimension data, if new values are in the taxi data, them will be loaded to it.

    """

    # Drop duplicates

    todays_dim_data = pd.DataFrame(taxi_trips[value_col].unique(), columns=[value_col])

    # Then find all those values which are not included already in dim table

    new_dim_data = todays_dim_data[~todays_dim_data[value_col].isin(dim_df[value_col])]

    # Then giving new ID to the new value (WITHOUT OVERWRITING ALREADY EXISTING IDs!!!) (based on 06 notebook)

    if not new_dim_data.empty:
        max_id = dim_df[id_col].max()
        new_dim_data[id_col] = range(max_id+1,max_id+1+len(new_dim_data))
        dim_df = pd.concat([dim_df, new_dim_data], ignore_index = True )

    return dim_df

In [ ]:
# Connecting companies with company IDs (numbers starting from 1) (based on 06 notebook)

dim_payment_type = taxi_trips['payment_type'].drop_duplicates().reset_index(drop=True)

# Then creating df

dim_payment_type = pd.DataFrame(

    {
        "payment_type_id" : range(1,len(dim_payment_type)+1),
        "payment_type": dim_payment_type
    }
)


# DUMMY DF GENERATION --> preparing the code for upcoming updates and potential new payment types

# Create basic structure: list(dict)

dummy_payment_type_data = [

{'payment_type': 'Credit Card'},
{'payment_type': 'X'},
{'payment_type': 'Y'},
{'payment_type': 'X'},

]

dummy_payment_type_data_df = pd.DataFrame(dummy_payment_type_data)

dim_company = taxi_trips['company'].drop_duplicates().reset_index(drop=True)

# Making new dataframe

dim_company = pd.DataFrame(

    {
        "company_id" : range(1,len(dim_company)+1),
        "company": dim_company
    }
)


dummy_company_data = [
    {"company":"Metro Jet Taxi A."},
    {"company":"X"},
    {"company":"Y"},
    {"company":"X"}
]

dummy_company_data_df = pd.DataFrame(dummy_company_data)

In [19]:
dim_payment_type_udpated = update_dim_company_dim_payment_type(taxi_trips, dim_payment_type,"payment_type_id", "payment_type" )
dim_company_type_udpated = update_dim_company_dim_payment_type(taxi_trips, dim_company,"company_id", "company" )

In [ ]:
# Check

dim_payment_type_udpated

In [ ]:
# Check

dim_company_type_udpated

## Update fact_taxi_trips with company and payment_type ids

In [20]:
def update_fact_taxi_trips_with_dimension_data(taxi_trips: pd.DataFrame, dim_payment_type: pd.DataFrame, dim_company: pd.DataFrame) -> pd.DataFrame:
    """ Upadte the fact_taxi_trips DataFrame with the company_master and payment_type ids and delete the string columns.
    Args:
        taxi_trips (pd.DataFrame):          The dataframe with the daily taxi trips.
        dim_payment_type (pd.DataFrame):    The payment type dimension table.
        dim_company (pd.DataFrame):         The company dimension table.
    Returns:
        pd.DataFrame:                       The taxi trips data with only payment_type_id and company_id without company or payment_type values.
    """
    
    fact_taxi_trips = taxi_trips.merge(dim_payment_type, on='payment_type')
    fact_taxi_trips = fact_taxi_trips.merge(dim_company, on='company')
    fact_taxi_trips.drop(['payment_type','company'], axis=1, inplace=True)

    return fact_taxi_trips

In [21]:
taxi_trips_transformed_with_dim_ids = update_fact_taxi_trips_with_dimension_data(taxi_trips_transformed,dim_payment_type,dim_company)
taxi_trips_transformed_with_dim_ids.head()

,trip_id,taxi_id,trip_start_timestamp,trip_end_timestamp,trip_seconds,trip_miles,pickup_community_area_id,dropoff_community_area_id,fare,tips,tolls,extras,trip_total,pickup_centroid_latitude,pickup_centroid_longitude,dropoff_centroid_latitude,dropoff_centroid_longitude,datetime_for_weather,payment_type_id,company_id
0,64451d9e83f94dc7cd493a58b21907f723a4d11c,48d462bcf24c2aadcda3fcf689d3f63fc178f23dc73bba...,2025-09-30 23:45:00,2025-10-01T00:15:00.000,1563,17.51,76,8,43.25,9.55,0,4,57.3,41.980264315,-87.913624596,41.899602111,-87.633308037,2025-09-30 23:00:00,1,1
1,013cfcf119425927ffaa918d90c6fa14e99fb66f,73b2f5adecea91eeef3900303a07f1b0519a594cffb6b0...,2025-09-30 23:45:00,2025-10-01T00:00:00.000,480,1.32,8,32,7.67,0,0,0,8.17,41.899602111,-87.633308037,41.878865584,-87.625192142,2025-09-30 23:00:00,2,2
2,0c93fd68ba8a33329f43b5ecd06c9adb4bd6c1ca,63d895bf335c522af83a8f2c608e31bb46a5d78cde4df2...,2025-09-30 23:45:00,2025-10-01T00:15:00.000,1475,18.34,76,8,45.25,11.15,0,10,66.9,41.980264315,-87.913624596,41.899602111,-87.633308037,2025-09-30 23:00:00,2,3
3,0fb91fa81f44cfd13beef5a9eab75da222dd3214,99ec13d5d806f5f5fa7a57910f8e38d84f90630529f2f8...,2025-09-30 23:45:00,2025-10-01T00:00:00.000,249,0.55,32,8,5,0,0,0,5,41.878865584,-87.625192142,41.899602111,-87.633308037,2025-09-30 23:00:00,3,4
4,1528b5af92997e65634466efa11c894379843989,58a93192d6a7e977c8c9b10d36ed0e52325afc4b5ece70...,2025-09-30 23:45:00,2025-10-01T00:00:00.000,906,5.65,8,5,15.65,4.02,0,0,20.17,41.899602111,-87.633308037,41.947791586,-87.683834942,2025-09-30 23:00:00,2,5


## Weather transformation

In [ ]:
def transform_weather(weather:Dict) -> pd.DataFrame:
    """Select and transform weather data.

    :param weather: The daily weather data from the Open Meteo API
    :return:        Transformed weather pandas DataFrame.
    """
    weather_data = {
        "datetime" : data["hourly"]["time"],
        "temperature" : data["hourly"]["temperature_2m"],
        "wind_speed" : data["hourly"]["wind_speed_10m"],
        "rain" : data["hourly"]["rain"],
        "precipitation" : data["hourly"]["precipitation"]
    }

    weather_df = pd.DataFrame(weather_data)

    # Change datetime datatype from object to datetime

    weather_df["datetime"] = pd.to_datetime(weather_df["datetime"])

    return weather_df

In [ ]:
current_datetime = datetime.now() - relativedelta(months=2)

formatted_datetime = current_datetime.strftime("%Y-%m-%d")

url = "https://archive-api.open-meteo.com/v1/era5"


# URL update based on variables with dictionary

params = {
    "latitude": 41.85,
    "longitude": -87.65,
    "start_date": formatted_datetime,
    "end_date": formatted_datetime,
    "hourly" : "temperature_2m,wind_speed_10m,rain,precipitation"
}

response = requests.get(url , params=params)
weather_raw_data = response.json()

In [ ]:
weather_df = transform_weather(weather_raw_data)

# Check

weather_df.head()

,datetime,temperature,wind_speed,rain,precipitation
0,2025-09-30 00:00:00,22.6,5.3,0.0,0.0
1,2025-09-30 01:00:00,21.7,5.2,0.0,0.0
2,2025-09-30 02:00:00,21.3,5.8,0.0,0.0
3,2025-09-30 03:00:00,20.9,4.5,0.0,0.0
4,2025-09-30 04:00:00,20.5,4.0,0.0,0.0
